## Rolling Window (200h cap)

In [7]:
# ============================================================
# Cell 1+2 combined: Extract vitals from chartevents zip
# and save vitals_full_200h.csv in one uninterrupted run
# ============================================================
import numpy as np
import pandas as pd
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")
DATA_DIR   = Path("C:/Users/20220505/Desktop/input path")
MAX_HOURS  = 200

VITAL_ITEMIDS = {
    220045: 'heart_rate',
    220179: 'abp_sys',
    220180: 'abp_dia',
    220181: 'abp_mean',
    220210: 'resp_rate',
    220277: 'spo2',
    223761: 'temp_f',
    223762: 'temp_c',
    220050: 'abp_sys',
    220051: 'abp_dia',
    220052: 'abp_mean',
}
VITAL_COLS = ['heart_rate', 'abp_sys', 'abp_dia',
              'abp_mean', 'resp_rate', 'spo2', 'temp_c']
TARGET_IDS = set(VITAL_ITEMIDS.keys())

# ── Load cohort ───────────────────────────────────────────────
print("Loading cohort...")
cohort = pd.read_csv(
    str(OUTPUT_DIR / 'icu_cohort (1).csv'),
    usecols=['stay_id', 'intime']
)
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort_stays     = set(cohort['stay_id'].tolist())
stay_intime      = cohort.set_index('stay_id')['intime'].to_dict()
print(f"Cohort stays : {len(cohort_stays):,}")

# ── Stream chartevents in chunks ──────────────────────────────
print("\nStreaming chartevents.csv from zip...")
print("Filtering to cohort stays + vital itemids only")
print("Expected time: 5-15 minutes\n")

CHUNK_SIZE   = 500_000
chunks_read  = 0
rows_kept    = 0
vital_chunks = []

with zipfile.ZipFile(
    str(DATA_DIR / 'chartevents_labevents.zip'), 'r'
) as z:
    with z.open('Cleaned/chartevents.csv') as f:
        reader = pd.read_csv(
            f,
            usecols=['stay_id', 'charttime', 'itemid', 'valuenum'],
            chunksize=CHUNK_SIZE,
            dtype={'itemid': 'int32', 'valuenum': 'float32'},
            parse_dates=['charttime'],
        )
        for chunk in reader:
            chunks_read += 1

            chunk = chunk[
                chunk['stay_id'].isin(cohort_stays) &
                chunk['itemid'].isin(TARGET_IDS) &
                chunk['valuenum'].notna()
            ]

            if len(chunk) > 0:
                vital_chunks.append(chunk)
                rows_kept += len(chunk)

            if chunks_read % 20 == 0:
                print(f"  Chunks: {chunks_read:>4} | "
                      f"Rows kept: {rows_kept:>8,}", end='\r')

print(f"\nDone. Chunks read : {chunks_read:,}")
print(f"Vital rows kept   : {rows_kept:,}")

# ── Combine and process ───────────────────────────────────────
print("\nProcessing extracted vitals...")
vitals_raw = pd.concat(vital_chunks, ignore_index=True)
print(f"Combined shape : {vitals_raw.shape}")

# Map itemid → vital name
vitals_raw['vital'] = vitals_raw['itemid'].map(VITAL_ITEMIDS)

# Fahrenheit → Celsius
mask_f = vitals_raw['vital'] == 'temp_f'
vitals_raw.loc[mask_f, 'valuenum'] = (
    (vitals_raw.loc[mask_f, 'valuenum'] - 32) * 5 / 9
).astype('float32')
vitals_raw.loc[mask_f, 'vital'] = 'temp_c'

# Compute hour offset from intime
vitals_raw['intime'] = vitals_raw['stay_id'].map(stay_intime)
vitals_raw['hour']   = (
    (vitals_raw['charttime'] - vitals_raw['intime'])
    .dt.total_seconds() / 3600
).astype(int)

# Filter to valid hours
vitals_raw = vitals_raw[
    (vitals_raw['hour'] >= 0) &
    (vitals_raw['hour'] < MAX_HOURS)
].copy()

print(f"Rows after hour filter : {len(vitals_raw):,}")
print(f"Hour range             : {vitals_raw['hour'].min()} "
      f"— {vitals_raw['hour'].max()}")

# Aggregate to hourly means
vitals_hourly = (
    vitals_raw
    .groupby(['stay_id', 'hour', 'vital'])['valuenum']
    .mean()
    .reset_index()
)

# Pivot to wide
vitals_wide = vitals_hourly.pivot_table(
    index=['stay_id', 'hour'],
    columns='vital',
    values='valuenum',
    aggfunc='mean'
).reset_index()
vitals_wide.columns.name = None

# Ensure all 7 vital columns present
for col in VITAL_COLS:
    if col not in vitals_wide.columns:
        vitals_wide[col] = np.nan

vitals_wide = vitals_wide[['stay_id', 'hour'] + VITAL_COLS]

print(f"\nVitals wide shape : {vitals_wide.shape}")
print(f"Stays covered     : {vitals_wide['stay_id'].nunique():,}")
print(f"Hour range        : {vitals_wide['hour'].min()} "
      f"— {vitals_wide['hour'].max()}")

stays_beyond_24 = (
    vitals_wide[vitals_wide['hour'] > 24]['stay_id'].nunique()
)
print(f"Stays with hour > 24 : {stays_beyond_24:,}")

print(f"\nMissingness per column:")
for col in VITAL_COLS:
    miss = vitals_wide[col].isna().mean()
    print(f"  {col:<15} {miss:.2%} missing")

# ── Save immediately ──────────────────────────────────────────
vitals_wide.to_csv(
    str(OUTPUT_DIR / 'vitals_full_200h.csv'), index=False
)
print(f"\nSaved → vitals_full_200h.csv ✓")
print(f"File size : "
      f"{(OUTPUT_DIR / 'vitals_full_200h.csv').stat().st_size / 1e6:.1f} MB")
print("\nNow run the recovery cell (Cell 4a)")

Loading cohort...
Cohort stays : 94,458

Streaming chartevents.csv from zip...
Filtering to cohort stays + vital itemids only
Expected time: 5-15 minutes

  Chunks:  860 | Rows kept: 53,434,006
Done. Chunks read : 866
Vital rows kept   : 53,806,854

Processing extracted vitals...
Combined shape : (53806854, 4)
Rows after hour filter : 43,024,180
Hour range             : 0 — 199

Vitals wide shape : (6209530, 9)
Stays covered     : 94,417
Hour range        : 0 — 199
Stays with hour > 24 : 71,100

Missingness per column:
  heart_rate      0.90% missing
  abp_sys         8.31% missing
  abp_dia         8.32% missing
  abp_mean        8.25% missing
  resp_rate       2.12% missing
  spo2            3.05% missing
  temp_c          69.98% missing

Saved → vitals_full_200h.csv ✓
File size : 289.5 MB

Now run the recovery cell (Cell 4a)


In [9]:
# ============================================================
# Cell 4a: Recovery — reload all intermediates from disk
# Run after kernel restart or if variables were lost
# ============================================================
import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")
DATA_DIR   = Path("C:/Users/20220505/Desktop/input path")
MAX_HOURS  = 200

VITAL_COLS = ['heart_rate', 'abp_sys', 'abp_dia',
              'abp_mean', 'resp_rate', 'spo2', 'temp_c']

# ── Cohort ────────────────────────────────────────────────────
print("Loading cohort...")
cohort = pd.read_csv(
    str(OUTPUT_DIR / 'icu_cohort (1).csv'),
    usecols=['stay_id', 'intime']
)
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort_stays     = set(cohort['stay_id'].tolist())
stay_intime      = cohort.set_index('stay_id')['intime'].to_dict()
print(f"  Cohort stays : {len(cohort_stays):,}")

# ── Stay order + feature names ────────────────────────────────
print("Loading metadata...")
stay_ids_order = pd.read_csv(
    str(OUTPUT_DIR / 'stay_ids_order.csv')
).squeeze().tolist()
n_stays = len(stay_ids_order)

with open(str(OUTPUT_DIR / 'rich_feature_names.txt')) as f:
    rich_feature_names = f.read().splitlines()

print(f"  Stays in tensor : {n_stays:,}")
print(f"  Rich features   : {len(rich_feature_names)}")

# ── Vitals full 200h ──────────────────────────────────────────
print("\nLoading vitals_full_200h.csv...")
vitals_wide = pd.read_csv(str(OUTPUT_DIR / 'vitals_full_200h.csv'))
print(f"  vitals_wide : {vitals_wide.shape}")
print(f"  Hour range  : {vitals_wide['hour'].min()} "
      f"— {vitals_wide['hour'].max()}")

# ── Urine output ──────────────────────────────────────────────
print("\nBuilding urine output hourly...")
uo = pd.read_csv(
    str(OUTPUT_DIR / 'urine_output_filtered.csv'),
    parse_dates=['charttime']
)
uo = uo[uo['stay_id'].isin(cohort_stays)].copy()
uo['intime'] = uo['stay_id'].map(stay_intime)
uo['hour']   = (
    (uo['charttime'] - uo['intime'])
    .dt.total_seconds() / 3600
).astype(int)
uo = uo[(uo['hour'] >= 0) & (uo['hour'] < MAX_HOURS)]
uo_hourly = (
    uo.groupby(['stay_id', 'hour'])['value']
    .sum()
    .reset_index()
    .rename(columns={'value': 'urine_output'})
)
print(f"  uo_hourly : {uo_hourly.shape}")
print(f"  Hour range: {uo_hourly['hour'].min()} "
      f"— {uo_hourly['hour'].max()}")

# ── Vasopressors ──────────────────────────────────────────────
print("\nBuilding vasopressors hourly...")
vp = pd.read_csv(
    str(OUTPUT_DIR / 'vasopressors_filtered.csv'),
    parse_dates=['starttime']
)
vp = vp[vp['stay_id'].isin(cohort_stays)].copy()
vp['intime'] = vp['stay_id'].map(stay_intime)
vp['hour']   = (
    (vp['starttime'] - vp['intime'])
    .dt.total_seconds() / 3600
).astype(int)
vp = vp[(vp['hour'] >= 0) & (vp['hour'] < MAX_HOURS)]
vp_hourly = (
    vp.groupby(['stay_id', 'hour'])
    .size()
    .reset_index(name='vasopressor_flag')
)
vp_hourly['vasopressor_flag'] = 1
print(f"  vp_hourly : {vp_hourly.shape}")

# ── Extra labs ────────────────────────────────────────────────
print("\nBuilding labs hourly...")
LAB_ITEMIDS = {
    51265: 'platelets_raw',
    51301: 'wbc',
    51237: 'inr',
    50813: 'lactate',
    50889: 'crp',
}
el = pd.read_csv(
    str(OUTPUT_DIR / 'extra_labs_filtered.csv'),
    parse_dates=['charttime']
)
el = el[
    el['stay_id'].isin(cohort_stays) &
    el['itemid'].isin(LAB_ITEMIDS)
].copy()
el['vital']  = el['itemid'].map(LAB_ITEMIDS)
el['intime'] = el['stay_id'].map(stay_intime)
el['hour']   = (
    (el['charttime'] - el['intime'])
    .dt.total_seconds() / 3600
).astype(int)
el = el[(el['hour'] >= 0) & (el['hour'] < MAX_HOURS)]
labs_hourly = (
    el.groupby(['stay_id', 'hour', 'vital'])['valuenum']
    .mean()
    .reset_index()
    .pivot_table(
        index=['stay_id', 'hour'],
        columns='vital',
        values='valuenum',
        aggfunc='mean'
    )
    .reset_index()
)
labs_hourly.columns.name = None
print(f"  labs_hourly : {labs_hourly.shape}")
print(f"  Lab columns : {list(labs_hourly.columns)}")

# ── SOFA labs ─────────────────────────────────────────────────
print("\nBuilding SOFA labs hourly...")
sl = pd.read_csv(str(OUTPUT_DIR / 'sofa_labs_hourly_wide.csv'))
sl = sl[sl['stay_id'].isin(cohort_stays)].copy()
sl = sl.rename(columns={'charttime_hour': 'charttime'})
sl['charttime'] = pd.to_datetime(sl['charttime'])
sl['intime']    = sl['stay_id'].map(stay_intime)
sl['hour']      = (
    (sl['charttime'] - sl['intime'])
    .dt.total_seconds() / 3600
).astype(int)
sl = sl[(sl['hour'] >= 0) & (sl['hour'] < MAX_HOURS)]
sl_hourly = (
    sl.groupby(['stay_id', 'hour'])[
        ['bilirubin', 'creatinine', 'platelets']
    ].mean().reset_index()
)
print(f"  sl_hourly : {sl_hourly.shape}")

# ── Build stay-hour index ─────────────────────────────────────
print("\nBuilding stay-hour index...")
all_stay_hours = pd.DataFrame({
    'stay_id': np.repeat(stay_ids_order, MAX_HOURS),
    'hour'   : np.tile(np.arange(MAX_HOURS), n_stays)
})
print(f"  all_stay_hours : {all_stay_hours.shape}")

# ── Merge all sources ─────────────────────────────────────────
print("\nMerging all feature sources...")
df = all_stay_hours.merge(
    vitals_wide, on=['stay_id', 'hour'], how='left'
)
df = df.merge(uo_hourly,  on=['stay_id', 'hour'], how='left')
df = df.merge(vp_hourly,  on=['stay_id', 'hour'], how='left')
df['vasopressor_flag'] = (
    df['vasopressor_flag'].fillna(0).astype('float32')
)
df = df.merge(labs_hourly, on=['stay_id', 'hour'], how='left')
df = df.merge(sl_hourly,   on=['stay_id', 'hour'], how='left')
print(f"  Merged df : {df.shape}")

# ── Rename SOFA columns ───────────────────────────────────────
df = df.rename(columns={
    'platelets'  : 'sofa_platelets',
    'bilirubin'  : 'sofa_bilirubin',
    'creatinine' : 'sofa_creatinine',
})

# ── Derived features ──────────────────────────────────────────
print("\nComputing derived features...")
df['shock_index']    = (
    df['heart_rate'] / (df['abp_sys'] + 1e-6)
).astype('float32')
df['hr_delta']       = (
    df.groupby('stay_id')['heart_rate'].diff()
).astype('float32')
df['temp_deviation'] = (df['temp_c'] - 37.0).astype('float32')

for col, flag in [
    ('lactate',      'lactate_fresh'),
    ('wbc',          'wbc_fresh'),
    ('crp',          'crp_fresh'),
    ('platelets_raw','platelets_fresh'),
    ('inr',          'inr_fresh'),
]:
    df[flag] = (
        df[col].notna().astype('float32')
        if col in df.columns else 0.0
    )

# ── LOCF imputation ───────────────────────────────────────────
print("Applying LOCF imputation per stay...")
df = df.sort_values(['stay_id', 'hour']).reset_index(drop=True)
fill_cols = [
    c for c in rich_feature_names
    if c in df.columns and not c.endswith('_fresh')
]
df[fill_cols] = (
    df.groupby('stay_id')[fill_cols]
    .transform(lambda x: x.ffill().bfill())
)

# ── Median fill ───────────────────────────────────────────────
print("Applying median fill...")
MEDIANS = {
    'abp_dia'        : 65.0,
    'abp_mean'       : 85.0,
    'abp_sys'        : 115.0,
    'heart_rate'     : 80.0,
    'resp_rate'      : 18.0,
    'spo2'           : 97.0,
    'temp_c'         : 37.0,
    'lactate'        : 1.5,
    'wbc'            : 8.0,
    'crp'            : 5.0,
    'platelets_raw'  : 200.0,
    'inr'            : 1.1,
    'urine_output'   : 60.0,
    'vasopressor_flag': 0.0,
    'shock_index'    : 0.7,
    'hr_delta'       : 0.0,
    'temp_deviation' : 0.0,
    'sofa_platelets' : 200.0,
    'sofa_bilirubin' : 0.0,
    'sofa_creatinine': 1.0,
}
for col, med in MEDIANS.items():
    if col in df.columns:
        df[col] = df[col].fillna(med).astype('float32')
for col in df.columns:
    if col.endswith('_fresh'):
        df[col] = df[col].fillna(0.0).astype('float32')

# ── Check all 25 features present ────────────────────────────
missing = [c for c in rich_feature_names if c not in df.columns]
if missing:
    print(f"\nMissing features — zero-filling: {missing}")
    for c in missing:
        df[c] = 0.0
else:
    print("\nAll 25 features present ✓")

# ── Alert tier thresholds ─────────────────────────────────────
ALERT_THRESHOLDS = {
    'no_alert'  : (0.00, 0.10),   # score < 0.10
    'high_risk' : (0.10, 0.50),   # 0.10 ≤ score < 0.50
    'critical'  : (0.50, 1.00),   # score ≥ 0.50
}

ALERT_COLORS = {
    'no_alert'  : '#27AE60',   # green
    'high_risk' : '#E67E22',   # amber
    'critical'  : '#C0392B',   # red
}

HERO_HORIZON_H = 12

def get_alert_tier(score):
    """
    Classify a risk score into one of three alert tiers.
    
    Parameters
    ----------
    score : float — predicted probability of sepsis within 12h
    
    Returns
    -------
    str : 'no_alert' | 'high_risk' | 'critical'
    """
    if score >= ALERT_THRESHOLDS['critical'][0]:
        return 'critical'
    elif score >= ALERT_THRESHOLDS['high_risk'][0]:
        return 'high_risk'
    else:
        return 'no_alert'

# ── Verify ────────────────────────────────────────────────────
print("Alert tiers defined:")
for tier, (lo, hi) in ALERT_THRESHOLDS.items():
    color = ALERT_COLORS[tier]
    print(f"  {tier:<12} : {lo:.2f} ≤ score < {hi:.2f}   "
          f"color={color}")

# Quick sanity check
test_scores = [0.05, 0.15, 0.55, 0.99]
print("\nSanity check:")
for s in test_scores:
    print(f"  score={s:.2f} → {get_alert_tier(s)}")

Loading cohort...
  Cohort stays : 94,458
Loading metadata...
  Stays in tensor : 74,550
  Rich features   : 25

Loading vitals_full_200h.csv...
  vitals_wide : (6209530, 9)
  Hour range  : 0 — 199

Building urine output hourly...
  uo_hourly : (3320197, 3)
  Hour range: 0 — 199

Building vasopressors hourly...
  vp_hourly : (410709, 3)

Building labs hourly...
  labs_hourly : (939002, 7)
  Lab columns : ['stay_id', 'hour', 'crp', 'inr', 'lactate', 'platelets_raw', 'wbc']

Building SOFA labs hourly...
  sl_hourly : (845002, 5)

Building stay-hour index...
  all_stay_hours : (14910000, 2)

Merging all feature sources...
  Merged df : (14910000, 19)

Computing derived features...
Applying LOCF imputation per stay...
Applying median fill...

All 25 features present ✓
Alert tiers defined:
  no_alert     : 0.00 ≤ score < 0.10   color=#27AE60
  high_risk    : 0.10 ≤ score < 0.50   color=#E67E22
  critical     : 0.50 ≤ score < 1.00   color=#C0392B

Sanity check:
  score=0.05 → no_alert
  scor

In [10]:
# ============================================================
# Cell 4b: Build and save X_rich_full.npy
# Shape: (74550, 200, 25)
# ============================================================
print("Building X_rich_full tensor...")
print(f"  Stays    : {n_stays:,}")
print(f"  Hours    : {MAX_HOURS}")
print(f"  Features : {len(rich_feature_names)}")

est_gb = n_stays * MAX_HOURS * len(rich_feature_names) * 4 / 1e9
print(f"  Est. size: {est_gb:.2f} GB")

# ── Allocate tensor ───────────────────────────────────────────
X_full = np.zeros(
    (n_stays, MAX_HOURS, len(rich_feature_names)),
    dtype=np.float32
)

stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}

# ── Fill tensor row by row ────────────────────────────────────
print("\nFilling tensor...")
processed = 0
skipped   = 0

for stay_id, group in df.groupby('stay_id'):
    if stay_id not in stay_to_idx:
        skipped += 1
        continue

    idx   = stay_to_idx[stay_id]
    hrs   = group['hour'].values.astype(int)
    vals  = group[rich_feature_names].values.astype(np.float32)
    valid = (hrs >= 0) & (hrs < MAX_HOURS)

    X_full[idx, hrs[valid], :] = vals[valid]
    processed += 1

    if processed % 10000 == 0:
        print(f"  Processed {processed:,} / {n_stays:,} stays...",
              end='\r')

print(f"\n  Stays processed : {processed:,}")
print(f"  Stays skipped   : {skipped:,}")

# ── Sanity checks ─────────────────────────────────────────────
print("\nSanity checks...")

# Compare hours 0-5 with original X_rich for first stay
X_orig = np.load(str(OUTPUT_DIR / 'X_rich.npy'))
print(f"  Original X_rich shape : {X_orig.shape}")
print(f"  X_rich_full shape     : {X_full.shape}")

print(f"\n  Feature 0 (abp_dia) — stay 0, hours 0-5:")
print(f"    Original   : {X_orig[0, :6, 0].round(2)}")
print(f"    X_rich_full: {X_full[0, :6, 0].round(2)}")

print(f"\n  Feature 3 (heart_rate) — stay 0, hours 0-5:")
print(f"    Original   : {X_orig[0, :6, 3].round(2)}")
print(f"    X_rich_full: {X_full[0, :6, 3].round(2)}")

# Coverage beyond hour 24
beyond_24 = X_full[:, 24:, :3].any(axis=2).any(axis=1)
print(f"\n  Stays with data beyond hour 24 : {beyond_24.sum():,} "
      f"({beyond_24.mean():.1%})")

# Coverage beyond hour 48
beyond_48 = X_full[:, 48:, :3].any(axis=2).any(axis=1)
print(f"  Stays with data beyond hour 48 : {beyond_48.sum():,} "
      f"({beyond_48.mean():.1%})")

# Non-zero rate
nonzero = (X_full != 0).mean()
print(f"  Non-zero entries               : {nonzero:.2%}")

# Check a long-stay patient at hour 50
long_stay_ids = (
    pd.read_csv(str(OUTPUT_DIR / 'hourly_labels.csv'),
                usecols=['stay_id', 'hour'])
    .groupby('stay_id')['hour'].max()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)
print(f"\n  Checking 3 longest stays at hour 50:")
for sid in long_stay_ids:
    if sid in stay_to_idx:
        idx  = stay_to_idx[sid]
        vals = X_full[idx, 50, :3]
        print(f"    stay={sid} hr50 "
              f"abp_dia={vals[0]:.1f} "
              f"abp_mean={vals[1]:.1f} "
              f"abp_sys={vals[2]:.1f}")

# ── Save ──────────────────────────────────────────────────────
print("\nSaving X_rich_full.npy...")
np.save(str(OUTPUT_DIR / 'X_rich_full.npy'), X_full)

fsize = (OUTPUT_DIR / 'X_rich_full.npy').stat().st_size / 1e9
print(f"Saved → X_rich_full.npy ✓")
print(f"Size   : {fsize:.2f} GB")
print(f"Shape  : {X_full.shape}")
print("\n03c complete ✓")
print("Ready for 03d — rolling inference")

Building X_rich_full tensor...
  Stays    : 74,550
  Hours    : 200
  Features : 25
  Est. size: 1.49 GB

Filling tensor...
  Processed 70,000 / 74,550 stays...
  Stays processed : 74,550
  Stays skipped   : 0

Sanity checks...
  Original X_rich shape : (74550, 24, 25)
  X_rich_full shape     : (74550, 200, 25)

  Feature 0 (abp_dia) — stay 0, hours 0-5:
    Original   : [72. 61. 65. 55. 56. 63.]
    X_rich_full: [75.5 66.5 66.5 60.  56.  63. ]

  Feature 3 (heart_rate) — stay 0, hours 0-5:
    Original   : [104.  83.  92.  83. 103. 111.]
    X_rich_full: [102.  102.   83.   87.5 103.  111. ]

  Stays with data beyond hour 24 : 74,546 (100.0%)
  Stays with data beyond hour 48 : 74,542 (100.0%)
  Non-zero entries               : 73.89%

  Checking 3 longest stays at hour 50:
    stay=39667768 hr50 abp_dia=80.0 abp_mean=90.0 abp_sys=123.0
    stay=37731486 hr50 abp_dia=100.5 abp_mean=128.5 abp_sys=185.5
    stay=38690919 hr50 abp_dia=53.0 abp_mean=77.0 abp_sys=152.0

Saving X_rich_full.n